# Система прогнозирования пиковых нагрузок — полный пайплайн

**ВКР: «Разработка системы проактивного масштабирования веб-приложений»**

| Секция | Глава ВКР | Что делает |
|--------|-----------|------------|
| §3.4 EDA | Глава 3 | Загрузка данных, разведочный анализ |
| §4.1 Модели | Глава 4 | XGBoost (Random Search) + LSTM (Random Search) + NeuralProphet (Grid Search) |
| §4.2 Пики | Глава 4 | Детекция пиковых нагрузок |
| §4.3 Walk-Forward | Глава 4 | Адаптивное переобучение с ADWIN-детектором дрейфа |

> **Запускай ячейки строго сверху вниз** (Runtime → Run all, или по одной)

## 0. Установка и клонирование

In [ ]:
import os, sys

REPO_DIR = '/content/Dyplom'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Roman-bib/Dyplom.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Добавляем в sys.path ЗДЕСЬ — это первая ячейка, всегда выполняется
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# numpy<2.0 нужен: NeuralProphet использует np.NaN, убранный в NumPy 2.0
!pip install -q xgboost neuralprophet tensorflow scikit-learn pandas "numpy<2.0" \
               matplotlib seaborn river joblib

print(f'Рабочая директория: {os.getcwd()}')
print(f'sys.path[0]: {sys.path[0]}')

## 1. Конфигурация — измени CSV_PATH под свои данные

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

# ================================================================
# НАСТРОЙКА — меняй только здесь
# ================================================================

CSV_PATH  = "/content/drive/MyDrive/Colab Notebooks/SDV_hourly.csv"
SAVE_DIR  = "/content/results"

# True = пропустить NeuralProphet (~15 мин экономии)
FAST_MODE = False

# Walk-Forward
WF_N_FRESH    = 200   # принудительный retrain каждые N шагов
WF_CONFIRM_N  = 10    # подтверждение ADWIN (шагов подряд)
WF_TEST_LIMIT = 500   # None = все точки теста

# ================================================================

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'CSV_PATH : {CSV_PATH}')
print(f'SAVE_DIR : {SAVE_DIR}')
print(f'FAST_MODE: {FAST_MODE}')

## 2. Монтирование Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §3.4 — Загрузка данных и EDA

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

import config as _cfg
from data_collection.csv_loader import load_csv

# Загружаем с экзогенными признаками (праздники, акции, кампании)
EXTRA = getattr(_cfg, "EXOG_COLS", [])
df = load_csv(CSV_PATH, extra_cols=EXTRA)

print(f'Загружено : {len(df)} точек')
print(f'Период    : {df["ds"].iloc[0]} → {df["ds"].iloc[-1]}')
print(f'min/max/mean: {df["y"].min():.0f} / {df["y"].max():.0f} / {df["y"].mean():.0f}')
print(f'NaN       : {df["y"].isna().sum()}')
print(f'Колонки   : {list(df.columns)}')

df.to_csv(f'{SAVE_DIR}/raw_data.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.plot(df['ds'], df['y'], linewidth=0.7, color='steelblue')
ax.set_title('Временной ряд нагрузки')
ax.set_ylabel('RPS'); ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(df['y'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(df['y'].mean(),          color='red',    linestyle='--', label=f'mean={df["y"].mean():.0f}')
ax.axvline(df['y'].quantile(0.95),  color='orange', linestyle='--', label=f'p95={df["y"].quantile(0.95):.0f}')
ax.set_title('Распределение нагрузки'); ax.set_xlabel('RPS')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eda_overview.png', dpi=150)
plt.show()

In [ ]:
df['hour']      = pd.to_datetime(df['ds']).dt.hour
df['dayofweek'] = pd.to_datetime(df['ds']).dt.dayofweek
dow_names = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly = df.groupby('hour')['y'].mean()
axes[0].bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
axes[0].set_title('Средняя нагрузка по часам суток')
axes[0].set_xlabel('Час'); axes[0].set_ylabel('RPS'); axes[0].grid(alpha=0.3, axis='y')

daily = df.groupby('dayofweek')['y'].mean()
axes[1].bar([dow_names[i] for i in daily.index], daily.values, color='orange', edgecolor='white')
axes[1].set_title('Средняя нагрузка по дням недели')
axes[1].set_ylabel('RPS'); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eda_seasonality.png', dpi=150)
plt.show()

df.drop(columns=['hour', 'dayofweek'], inplace=True)

## Разбиение train / val / test

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import pandas as pd
import matplotlib.pyplot as plt

from preprocessing.feature_engineering import split_train_val_test

n = len(df)
TEST_H = min(480, n // 6)
VAL_H  = TEST_H

train, val, test = split_train_val_test(df, test_hours=TEST_H, val_hours=VAL_H)
print(f'Train : {len(train)} ({train["ds"].iloc[0]} → {train["ds"].iloc[-1]})')
print(f'Val   : {len(val)} ({val["ds"].iloc[0]} → {val["ds"].iloc[-1]})')
print(f'Test  : {len(test)} ({test["ds"].iloc[0]} → {test["ds"].iloc[-1]})')

train.to_csv(f'{SAVE_DIR}/train.csv', index=False)
val.to_csv(f'{SAVE_DIR}/val.csv', index=False)
test.to_csv(f'{SAVE_DIR}/test.csv', index=False)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train['ds'], train['y'], color='steelblue', linewidth=0.7, label=f'Train ({len(train)})')
ax.plot(val['ds'],   val['y'],   color='green',     linewidth=0.7, label=f'Val ({len(val)})')
ax.plot(test['ds'],  test['y'],  color='red',        linewidth=0.7, label=f'Test ({len(test)})')
ax.set_title('Разбиение на train / val / test')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/data_split.png', dpi=150)
plt.show()

## §4.1 — Сравнение моделей (с подбором гиперпараметров)

- **XGBoost** — Random Search, 30 итераций, 7 параметров
- **LSTM** — Random Search, 10 итераций, 5 параметров, Huber loss
- **NeuralProphet** — Grid Search, 8 комбинаций (отключается `FAST_MODE=True`)

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import pandas as pd
from models.comparison import ModelComparison
from preprocessing.feature_engineering import FeatureBuilder, _infer_step_minutes, _hours_to_periods

step_min    = _infer_step_minutes(pd.to_datetime(train['ds']))
lstm_window = max(12, min(_hours_to_periods(24, step_min), 288))
print(f'Шаг данных: {step_min} мин  |  LSTM window_size: {lstm_window}')

comparator = ModelComparison(model_save_dir=SAVE_DIR)
results = comparator.run(
    train, val, test,
    include_prophet  = not FAST_MODE,
    include_lstm     = True,
    lstm_window_size = lstm_window,
    force_retrain    = True,
)

comparator.save_best()
best_name, best_model = comparator.best_model()
print(f'\nЛучшая модель: {best_name}')

In [ ]:
# Таблица метрик
import pandas as pd

SHOW = ['MAE', 'RMSE', 'MAPE', 'train_time_s']
rows = []
for name, m in comparator.results_.items():
    row = {'Модель': name}
    row.update({c: m.get(c, float('nan')) for c in SHOW})
    if 'mae_train' in m:
        row['MAE_train']  = m['mae_train']
        row['MAE_val']    = m['mae_val']
        row['overfit']    = m['overfit_ratio']
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index('Модель').round(3)
metrics_df.to_csv(f'{SAVE_DIR}/metrics_comparison.csv')
display(metrics_df)

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import matplotlib.pyplot as plt
from evaluation.metrics import plot_all_forecasts

plot_all_forecasts(train, val, test,
    predictions_dict=comparator.predictions_,
    save_path=f'{SAVE_DIR}/comparison_all_models.png', zoom=False)
plot_all_forecasts(train, val, test,
    predictions_dict=comparator.predictions_,
    save_path=f'{SAVE_DIR}/comparison_zoom.png', zoom=True)
plt.show()

### Важность признаков XGBoost

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import joblib, numpy as np
import matplotlib.pyplot as plt
from models.xgboost_model import feature_importance
from preprocessing.feature_engineering import FeatureBuilder

xgb_model = comparator.models_.get('XGBoost') or joblib.load(f'{SAVE_DIR}/xgboost.pkl')
builder   = FeatureBuilder()
X_tr_fi, _ = builder.get_X_y(train)

imp_df = feature_importance(xgb_model, feature_names=list(X_tr_fi.columns))
imp_df.to_csv(f'{SAVE_DIR}/feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
top12 = imp_df.head(12).iloc[::-1]
ax.barh(top12['feature'], top12['importance'], color='steelblue', edgecolor='white')
ax.set_title('Важность признаков XGBoost (gain), топ-12')
ax.set_xlabel('Importance'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/feature_importance.png', dpi=150)
plt.show()
print(imp_df.to_string(index=False))

### Доверительный интервал (квантильная регрессия 10–90%)

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import numpy as np
import matplotlib.pyplot as plt
from models.xgboost_model import get_confidence_interval
from evaluation.metrics import plot_forecast
from preprocessing.feature_engineering import FeatureBuilder

builder_ci = FeatureBuilder()
(X_tr, y_tr), (X_vl, y_vl), (X_te, y_te) = builder_ci.transform_splits(train, val, test)

print('Обучение квантильных моделей (q=0.10 и q=0.90)...')
lower, upper = get_confidence_interval(X_tr, y_tr, X_vl, y_vl, X_te, save_dir=SAVE_DIR)

xgb_preds = comparator.predictions_.get('XGBoost')
plot_forecast(train, val, test, xgb_preds, 'XGBoost',
              lower=lower, upper=upper,
              save_path=f'{SAVE_DIR}/forecast_xgb_ci.png', zoom=False)
plot_forecast(train, val, test, xgb_preds, 'XGBoost',
              lower=lower, upper=upper,
              save_path=f'{SAVE_DIR}/forecast_xgb_ci_zoom.png', zoom=True)
plt.show()

# Экспорт предсказаний с CI
pred_df = test[['ds','y']].iloc[:len(xgb_preds)].copy().reset_index(drop=True)
for name, p in comparator.predictions_.items():
    pred_df[name] = np.array(p[:len(pred_df)])
pred_df['XGBoost_lower'] = lower[:len(pred_df)]
pred_df['XGBoost_upper'] = upper[:len(pred_df)]
pred_df.to_csv(f'{SAVE_DIR}/predictions.csv', index=False)
print(f'Сохранено: {SAVE_DIR}/predictions.csv')

## §4.2 — Детекция пиковых нагрузок

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import pandas as pd
import numpy as np
from evaluation.peak_detection import PeakDetector
from preprocessing.feature_engineering import FeatureBuilder

rps_max = float(train['y'].max())
detector = PeakDetector(
    method='rolling_std', k=2.0,
    target_rps_per_replica=rps_max / 10,
    min_replicas=1, max_replicas=10,
)
detector.fit(train['y'])

builder_pk = FeatureBuilder()
X_te_pk, y_te_pk = builder_pk.get_X_y(test)
xgb_preds_pk = comparator.predictions_.get('XGBoost')
pred_series  = pd.Series(xgb_preds_pk[:len(y_te_pk)], index=y_te_pk.index)

events_df  = detector.detect_series(y_te_pk, pred_series)
summary_pk = detector.summary(events_df)

print(f'Порог  : {summary_pk["threshold"]:.0f} RPS')
print(f'Пиков  : {summary_pk["peaks_detected"]} / {summary_pk["total_points"]} ({summary_pk["peak_ratio_pct"]}%)')
print(f'Severity: {summary_pk["severity_counts"]}')

events_df.to_csv(f'{SAVE_DIR}/peak_events.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(events_df['timestamp'], events_df['rps'],       color='gray',      linewidth=0.7, label='Факт')
ax.plot(events_df['timestamp'], events_df['predicted'], color='steelblue', linewidth=0.9, label='Прогноз XGBoost')
ax.axhline(summary_pk['threshold'], color='red', linestyle='--', linewidth=1.0,
           label=f'Порог {summary_pk["threshold"]:.0f}')
peaks = events_df[events_df['is_peak']]
ax.scatter(peaks['timestamp'], peaks['rps'], color='red', s=12, zorder=5, label=f'Пики ({len(peaks)})')
ax.set_title('Детекция пиковых нагрузок')
ax.set_ylabel('RPS'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/peak_detection.png', dpi=150)
plt.show()

if peaks.shape[0] > 0:
    print('\nПервые 10 пиков:')
    display(peaks[['timestamp','rps','predicted','severity','recommended_replicas']].head(10))

## §4.3 — Walk-Forward валидация с адаптивным переобучением

Модель прогнозирует **точка за точкой**. При обнаружении концепт-дрейфа
(ADWIN или принудительно через `WF_N_FRESH` шагов) переобучается на всей
накопленной истории. Рядом идёт **фиксированная baseline-модель** для сравнения.

In [ ]:
import os, sys
REPO_DIR = '/content/Dyplom'
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

import joblib
from evaluation.walk_forward import run_walk_forward
from retraining.scheduler import make_xgb_train_fn
from retraining.drift_detector import ADWINDriftDetector
from models.forecasters import predict_xgboost_wf
from preprocessing.feature_engineering import FeatureBuilder

initial_xgb = comparator.models_.get('XGBoost') or joblib.load(f'{SAVE_DIR}/xgboost.pkl')
builder_wf  = FeatureBuilder()
drift_det   = ADWINDriftDetector(n_fresh=WF_N_FRESH, confirmation_n=WF_CONFIRM_N)
train_fn_wf = make_xgb_train_fn(builder_wf, save_dir=SAVE_DIR)
test_wf     = test.iloc[:WF_TEST_LIMIT].reset_index(drop=True) if WF_TEST_LIMIT else test.reset_index(drop=True)

print(f'Walk-forward: {len(test_wf)} шагов, n_fresh={WF_N_FRESH}, confirm={WF_CONFIRM_N}\n')

wf_res = run_walk_forward(
    train=train, val=val, test=test_wf,
    initial_model=initial_xgb,
    predict_fn=predict_xgboost_wf,
    train_fn=train_fn_wf,
    builder=builder_wf,
    drift_detector=drift_det,
    save_dir=SAVE_DIR,
    verbose=True,
)

In [ ]:
import numpy as np

res_df     = wf_res['results_df']
base_df    = wf_res['baseline_df']
summary_wf = wf_res['summary']
retrains   = summary_wf['retrain_timestamps']

print('=' * 50)
print('ИТОГ WALK-FORWARD')
print('=' * 50)
print(f'MAE адаптивная   : {summary_wf["mae_adaptive"]}')
print(f'MAE фиксированная: {summary_wf["mae_baseline"]}')
print(f'Улучшение        : {summary_wf["improvement_pct"]}%')
print(f'Переобучений     : {summary_wf["n_retrains"]}')
if summary_wf['retrain_intervals']:
    iv = summary_wf['retrain_intervals']
    print(f'Интервалы (шаги) : min={min(iv)}, max={max(iv)}, mean={np.mean(iv):.0f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax1 = axes[0]
ax1.plot(res_df['timestamp'],  res_df['y_true'],  color='gray',      linewidth=0.7, alpha=0.8, label='Факт')
ax1.plot(res_df['timestamp'],  res_df['y_pred'],  color='steelblue', linewidth=1.1, label='Адаптивная')
if 'y_pred_baseline' in base_df.columns:
    ax1.plot(base_df['timestamp'], base_df['y_pred_baseline'], color='orange', linewidth=1.0,
             linestyle='--', label='Фиксированная')
for i, ts in enumerate(retrains):
    ax1.axvline(ts, color='crimson', linestyle=':', linewidth=0.9, alpha=0.8,
                label='Переобучение' if i == 0 else None)
ax1.set_title('Walk-Forward: Прогноз vs Факт')
ax1.set_ylabel('RPS'); ax1.legend(loc='upper right'); ax1.grid(alpha=0.3)

ax2 = axes[1]
w = min(24, max(1, len(res_df) // 10))
roll_a = res_df['mae'].rolling(w, min_periods=1).mean()
roll_b = base_df['mae_baseline'].rolling(w, min_periods=1).mean()
ax2.plot(res_df['timestamp'],  roll_a, color='steelblue', linewidth=1.3, label=f'MAE адаптивная (rolling {w})')
ax2.plot(base_df['timestamp'], roll_b, color='orange',    linewidth=1.2, linestyle='--',
         label=f'MAE фиксированная (rolling {w})')
for ts in retrains:
    ax2.axvline(ts, color='crimson', linestyle=':', linewidth=0.9, alpha=0.7)
ax2.set_title('Скользящий MAE: адаптивная vs фиксированная')
ax2.set_ylabel('MAE'); ax2.set_xlabel('Время')
ax2.legend(loc='upper right'); ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=20)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/walk_forward_combined.png', dpi=150)
plt.show()
print(f'Сохранено: {SAVE_DIR}/walk_forward_combined.png')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Лог переобучений
log_path = f'{SAVE_DIR}/walk_forward_log.csv'
if os.path.exists(log_path):
    log_df = pd.read_csv(log_path)
    print(f'Событий переобучения: {len(log_df)}')
    display(log_df)

# Интервалы между переобучениями
if summary_wf['retrain_intervals']:
    iv = summary_wf['retrain_intervals']
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(len(iv)), iv, color='steelblue', edgecolor='white')
    ax.axhline(np.mean(iv), color='red', linestyle='--', label=f'Среднее: {np.mean(iv):.0f} шагов')
    ax.set_title('Интервалы между переобучениями')
    ax.set_xlabel('Номер переобучения'); ax.set_ylabel('Шагов')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/retrain_intervals.png', dpi=150)
    plt.show()

## §4.4 — Итоговая сводка и экспорт

In [ ]:
import os, pandas as pd, numpy as np

print('=' * 60)
print('ИТОГОВАЯ СВОДКА ЭКСПЕРИМЕНТА')
print('=' * 60)

print('\n§4.1  Сравнение моделей:')
display(metrics_df[['MAE','RMSE','MAPE']].round(2))

print(f'\n§4.2  Детекция пиков:')
print(f'  Порог: {summary_pk["threshold"]:.0f} RPS')
print(f'  Пиков: {summary_pk["peaks_detected"]} / {summary_pk["total_points"]} ({summary_pk["peak_ratio_pct"]}%)')

print(f'\n§4.3  Walk-Forward:')
print(f'  MAE адаптивная   : {summary_wf["mae_adaptive"]}')
print(f'  MAE фиксированная: {summary_wf["mae_baseline"]}')
print(f'  Улучшение        : {summary_wf["improvement_pct"]}%')
print(f'  Переобучений     : {summary_wf["n_retrains"]}')

summary_row = {
    'best_model':         best_name,
    **{f'{n}_mae': v.get('MAE') for n, v in comparator.results_.items()},
    'peak_threshold':     summary_pk['threshold'],
    'peaks_detected':     summary_pk['peaks_detected'],
    'wf_mae_adaptive':    summary_wf['mae_adaptive'],
    'wf_mae_baseline':    summary_wf['mae_baseline'],
    'wf_improvement_pct': summary_wf['improvement_pct'],
    'wf_n_retrains':      summary_wf['n_retrains'],
}
pd.DataFrame([summary_row]).to_csv(f'{SAVE_DIR}/experiment_summary.csv', index=False)

print(f'\nВсе файлы в {SAVE_DIR}/')
for f in sorted(os.listdir(SAVE_DIR)):
    kb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f'  {f:<45} {kb:.1f} KB')